# Day 4: Regex, JSON, CSV and working with API data (Exercises)

**Python for Everybody:** Course 3 *Using Python to Access Web Data*, chapters 11 (Regular Expressions), 12 (Network programs) and 13 (Web services: XML, JSON, APIs). All the examples here use text in memory, so no internet is needed.

**Why it matters for the JD:** 'Design APIs that expose data' and 'build data processing and integration components' means parsing JSON/CSV, validating records and handling API errors sensibly.

**Time:** about 90 minutes · 8 exercises

**How to use this notebook**
1. Run the first cell (it defines `check()`).
2. For each exercise: read the prompt, write your code in the code cell, run it, then run the check cell below it.
3. ✅ means it passed. ❌ prints a hint. Fix and re-run until green.
4. Don't open `solutions.ipynb` until you've tried. If stuck for 10 minutes, read only the hint.
5. Run the last cell for your score, then answer the *Say it out loud* questions.

In [ ]:
# Run this cell first. It gives you the check() helper used below.
_results = {}

def check(name, test):
    """Run a test function and print a pass/fail line with a hint."""
    try:
        test()
    except AssertionError as e:
        _results[name] = False
        print(f"❌ {name}: {e}")
    except Exception as e:
        _results[name] = False
        print(f"❌ {name}: {type(e).__name__}: {e}")
    else:
        _results[name] = True
        print(f"✅ {name}")

def raises(exc, fn, *args, **kwargs):
    """True if fn(*args, **kwargs) raises exc."""
    try:
        fn(*args, **kwargs)
    except exc:
        return True
    except Exception:
        return False
    return False

TOTAL = 8

def score():
    done = sum(_results.values())
    print(f"Score: {done}/{TOTAL}  " + ("🎉 all green!" if done == TOTAL else "keep going, re-run the cells you missed"))


This day uses the standard library only. Run the next cell, which imports what we need.

In [ ]:
import re, json, csv, io
from urllib.parse import urlencode

## Exercise 1: extract_numbers(text): regex findall  `warm-up`

The py4e regex exercise: return a **list of ints** for every run of digits in the text.

`extract_numbers("Order 12 had 3 items costing 450 total")` -> `[12, 3, 450]`

<details><summary>Hint</summary>

`re.findall(r"[0-9]+", text)` returns strings. Convert each with `int`.

</details>

In [ ]:
def extract_numbers(text):
    pass

In [ ]:
def _test():
    assert extract_numbers("Order 12 had 3 items costing 450 total") == [12, 3, 450]
    assert extract_numbers("no digits here") == []

check('1. extract_numbers(text): regex findall', _test)

## Exercise 2: find_emails(text)  `easy`

Return all email addresses found in the text, in order. Allow dots, `+` and `-` in the name part, and dotted domains.

Ignore fragments that aren't complete emails (like `@nope` or `me@`).

<details><summary>Hint</summary>

Something like `[\w.+-]+@[\w-]+(?:\.[\w-]+)+`.

</details>

In [ ]:
def find_emails(text):
    pass

In [ ]:
def _test():
    text = "Contact a.b@x.com or sales+uk@shop.co.in; not this: @nope or me@"
    assert find_emails(text) == ["a.b@x.com", "sales+uk@shop.co.in"]
    assert find_emails("nothing") == []

check('2. find_emails(text)', _test)

## Exercise 3: parse_log_line(line): regex groups  `medium`

Turn a log line into a dict, or return `None` if the line doesn't match the format.

`"2026-11-20 14:03:22 ERROR payment-service Timeout after 30s"` ->
```python
{"date": "2026-11-20", "time": "14:03:22", "level": "ERROR",
 "service": "payment-service", "message": "Timeout after 30s"}
```
Levels are `DEBUG`, `INFO`, `WARNING`, `ERROR`.

<details><summary>Hint</summary>

Use one compiled pattern with five capture groups: `(\d{4}-\d{2}-\d{2}) (\d{2}:\d{2}:\d{2}) (DEBUG|INFO|WARNING|ERROR) (\S+) (.+)`

</details>

In [ ]:
def parse_log_line(line):
    pass

In [ ]:
def _test():
    line = "2026-11-20 14:03:22 ERROR payment-service Timeout after 30s"
    assert parse_log_line(line) == {"date": "2026-11-20", "time": "14:03:22", "level": "ERROR", "service": "payment-service", "message": "Timeout after 30s"}
    assert parse_log_line("garbage line") is None
    assert parse_log_line("2026-11-20 14:03:22 FATAL svc boom") is None, "FATAL is not an allowed level"

check('3. parse_log_line(line): regex groups', _test)

## Exercise 4: names_of_active(json_text): parse JSON  `easy`

`json_text` is a JSON string like `{"users": [{"name": "Asha", "active": true}, ...]}`. Return the names of users whose `active` is true, in order. A user with no `active` key counts as inactive.

<details><summary>Hint</summary>

`json.loads` turns the text into dicts and lists. `user.get("active")` returns `None` if the key is missing.

</details>

In [ ]:
def names_of_active(json_text):
    pass

In [ ]:
def _test():
    text = '{"users": [{"name": "Asha", "active": true}, {"name": "Ravi", "active": false}, {"name": "Meena"}]}'
    assert names_of_active(text) == ["Asha"]
    assert names_of_active('{"users": []}') == []

check('4. names_of_active(json_text): parse JSON', _test)

## Exercise 5: flatten(d): nested dict to dotted keys  `medium`

Flatten a nested dict so inner keys become dotted paths.

`flatten({"a": {"b": 1, "c": {"d": 2}}, "e": 3})` -> `{"a.b": 1, "a.c.d": 2, "e": 3}`

This is how you prepare API data for a table or CSV.

<details><summary>Hint</summary>

Recursion: for each key, if the value is a dict, flatten it with the key as the new prefix; otherwise store it.

</details>

In [ ]:
def flatten(d, parent="", sep="."):
    pass

In [ ]:
def _test():
    assert flatten({"a": {"b": 1, "c": {"d": 2}}, "e": 3}) == {"a.b": 1, "a.c.d": 2, "e": 3}
    assert flatten({}) == {}
    assert flatten({"x": 1}) == {"x": 1}

check('5. flatten(d): nested dict to dotted keys', _test)

## Exercise 6: total_by_category(csv_text): the csv module  `medium`

`csv_text` has the header `category,amount`. Return a dict of the total amount per category, with totals rounded to 2 places.

```
category,amount
food,10.50
travel,200
food,4.25
```
-> `{"food": 14.75, "travel": 200.0}`

<details><summary>Hint</summary>

`csv.DictReader(io.StringIO(csv_text))` yields a dict per row. Amounts arrive as strings: use `float`.

</details>

In [ ]:
def total_by_category(csv_text):
    pass

In [ ]:
def _test():
    text = "category,amount\nfood,10.50\ntravel,200\nfood,4.25\n"
    assert total_by_category(text) == {"food": 14.75, "travel": 200.0}
    assert total_by_category("category,amount\n") == {}

check('6. total_by_category(csv_text): the csv module', _test)

## Exercise 7: build_url(base, params): query strings  `easy`

Add query parameters to a URL, properly encoded. With no params, return the base unchanged.

`build_url("https://api.x.com/items", {"q": "red shoes", "page": 2})` -> `"https://api.x.com/items?q=red+shoes&page=2"`

<details><summary>Hint</summary>

`urllib.parse.urlencode` (already imported above).

</details>

In [ ]:
def build_url(base, params):
    pass

In [ ]:
def _test():
    assert build_url("https://api.x.com/items", {"q": "red shoes", "page": 2}) == "https://api.x.com/items?q=red+shoes&page=2"
    assert build_url("https://api.x.com/items", {}) == "https://api.x.com/items"

check('7. build_url(base, params): query strings', _test)

## Exercise 8: Interview: handle_api_response(status, body_text)  `interview`

A client calls an API and gets a status code and a body. Write the handling logic (the two exception classes are given):
- `200`: parse and return the JSON. If the body is not valid JSON, raise `ValueError`.
- `404`: return `None`.
- `429` or any `5xx`: raise `RetryableError` (the caller may try again).
- any other status: raise `ClientError` (retrying won't help).

Interviewers use this to see whether you tell retryable from non-retryable failures.

<details><summary>Hint</summary>

`json.loads` raises `json.JSONDecodeError`, which is a subclass of `ValueError`.

</details>

In [ ]:
class ClientError(Exception):
    pass


class RetryableError(Exception):
    pass


def handle_api_response(status, body_text):
    pass

In [ ]:
def _test():
    assert handle_api_response(200, '{"a": 1}') == {"a": 1}
    assert handle_api_response(404, "") is None
    assert raises(RetryableError, handle_api_response, 429, "")
    assert raises(RetryableError, handle_api_response, 503, "")
    assert raises(ClientError, handle_api_response, 400, "")
    assert raises(ClientError, handle_api_response, 401, "")
    assert raises(ValueError, handle_api_response, 200, "not json")

check('8. Interview: handle_api_response(status, body_text)', _test)

## Your score

In [ ]:
score()

## Say it out loud (interview questions)

Answer each one aloud in under 60 seconds, as if a client asked. Model answers are in `solutions.ipynb`.

- What does `json.loads` vs `json.load` do?
- Regex: greedy vs non-greedy?
- How do you decide whether to retry a failed API call?
- Why use `csv.DictReader` instead of `line.split(',')`?
- How do you safely read a key that may be missing in a dict?
